In [1]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import (
    StateGraph,
    START,
    END
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from finagent.rag.pipeline import ask_rag
from finagent.market.pipeline import analyze_market
from finagent.forecast.pipeline import forecast_stock

Nombre de documents : 674


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index FAISS créé.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [2]:
# ============================================================
# 1. STATE
# ============================================================

class FinAgentState(TypedDict):

    question: str

    selected_agents: list[str]

    rag_answer: str
    market_answer: str
    forecast_answer: str

    final_answer: str


# ============================================================
# 2. ROUTER OUTPUT
# ============================================================

class RouteDecision(BaseModel):

    agents: list[
        Literal[
            "rag",
            "market",
            "forecast"
        ]
    ] = Field(
        min_length=1
    )

In [3]:
# ============================================================
# 3. ROUTER LLM
# ============================================================

router_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


structured_router = (
    router_llm.with_structured_output(
        RouteDecision
    )
)

In [4]:
# ============================================================
# 4. ROUTER PROMPT
# ============================================================

router_prompt = """
You are the router of FinAgent-X,
a multi-agent financial intelligence system
focused on Malaysian listed companies.

Select ONE OR MORE agents required
to answer the user's question.

Available agents:

rag:
Use for financial news, company events,
announcements, earnings reported in news,
management changes, acquisitions, contracts,
and explanations based on The Edge Malaysia.

market:
Use for current or historical market data
and technical indicators such as price,
returns, volume, RSI, MACD, moving averages,
volatility, momentum, ATR or Bollinger Bands.

forecast:
Use when the user asks about future price
direction, expected return, prediction,
forecast or outlook based on the forecasting model.


Examples:

"What did The Edge report about Maybank?"
→ ["rag"]

"What is Maybank's RSI?"
→ ["market"]

"Predict Maybank's next-day return."
→ ["forecast"]

"Explain Maybank's recent news and technical situation."
→ ["rag", "market"]

"Analyze Maybank using news, technical indicators
and the forecast."
→ ["rag", "market", "forecast"]


Select only the agents genuinely needed.

QUESTION:

{question}
"""

In [5]:
# ============================================================
# 5. ROUTER NODE
# ============================================================

def router_node(
    state: FinAgentState
):

    prompt = router_prompt.format(
        question=state["question"]
    )

    decision = structured_router.invoke(
        prompt
    )

    # Remove possible duplicates
    agents = list(
        dict.fromkeys(
            decision.agents
        )
    )

    return {
        "selected_agents":
            agents
    }

In [6]:
# ============================================================
# 6. RAG AGENT
# ============================================================

def rag_node(
    state: FinAgentState
):

    result = ask_rag(
        state["question"]
    )

    return {
        "rag_answer":
            result["answer"]
    }


# ============================================================
# 7. MARKET AGENT
# ============================================================

def market_node(
    state: FinAgentState
):

    result = analyze_market(
        state["question"]
    )

    return {
        "market_answer":
            result["answer"]
    }


# ============================================================
# 8. FORECAST AGENT
# ============================================================

def forecast_node(
    state: FinAgentState
):

    result = forecast_stock(
        state["question"]
    )

    return {
        "forecast_answer":
            result["answer"]
    }

In [7]:
# ============================================================
# 9. CONDITIONAL ROUTING
# ============================================================

def choose_agents(
    state: FinAgentState
):

    mapping = {
        "rag":
            "rag_agent",

        "market":
            "market_agent",

        "forecast":
            "forecast_agent"
    }

    return [
        mapping[agent]
        for agent
        in state["selected_agents"]
    ]


# ============================================================
# 10. SYNTHESIS LLM
# ============================================================

synthesis_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are the synthesis agent of FinAgent-X.

Your task is to combine the analyses produced
by specialized financial agents into one clear,
coherent answer.

Possible sources:

- News/RAG Agent:
  explains financial news and company events.

- Market Agent:
  interprets market data and technical indicators.

- Forecast Agent:
  provides an experimental model-based estimate
  of the next-session return.

Rules:

1. Use ONLY the agent outputs provided.

2. Do not invent financial facts.

3. Do not invent missing agent results.

4. Clearly distinguish:
   - news/events,
   - technical market signals,
   - forecasting estimates.

5. Forecasts are uncertain quantitative signals,
   not guaranteed future outcomes.

6. Answer the user's original question directly.

7. Be concise but sufficiently explanatory.
"""
        ),

        (
            "human",
            """
USER QUESTION:

{question}


SELECTED AGENTS:

{selected_agents}


NEWS / RAG ANALYSIS:

{rag_answer}


MARKET ANALYSIS:

{market_answer}


FORECAST ANALYSIS:

{forecast_answer}


FINAL SYNTHESIS:
"""
        )
    ]
)


synthesis_chain = (
    synthesis_prompt
    | synthesis_llm
    | StrOutputParser()
)

In [8]:
# ============================================================
# 11. SYNTHESIS NODE
# ============================================================

def synthesis_node(
    state: FinAgentState
):

    final_answer = synthesis_chain.invoke(
        {
            "question":
                state["question"],

            "selected_agents":
                ", ".join(
                    state[
                        "selected_agents"
                    ]
                ),

            "rag_answer":
                state["rag_answer"],

            "market_answer":
                state["market_answer"],

            "forecast_answer":
                state["forecast_answer"]
        }
    )

    return {
        "final_answer":
            final_answer
    }


In [9]:
# ============================================================
# 12. BUILD GRAPH
# ============================================================

builder = StateGraph(
    FinAgentState
)


builder.add_node(
    "router",
    router_node
)

builder.add_node(
    "rag_agent",
    rag_node
)

builder.add_node(
    "market_agent",
    market_node
)

builder.add_node(
    "forecast_agent",
    forecast_node
)

builder.add_node(
    "synthesis",
    synthesis_node
)


# START → ROUTER

builder.add_edge(
    START,
    "router"
)


# ROUTER → ONE OR MULTIPLE AGENTS

builder.add_conditional_edges(
    "router",
    choose_agents
)


# ALL SPECIALIZED AGENTS → SYNTHESIS

builder.add_edge(
    "rag_agent",
    "synthesis"
)

builder.add_edge(
    "market_agent",
    "synthesis"
)

builder.add_edge(
    "forecast_agent",
    "synthesis"
)


# SYNTHESIS → END

builder.add_edge(
    "synthesis",
    END
)


# ============================================================
# 13. COMPILE
# ============================================================

graph = builder.compile()

In [10]:
print(
    graph.get_graph().draw_ascii()
)

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | router |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [11]:
initial_state = {
    "question": (
        "Predict Maybank's return "
        "for next week."
    ),
    "route": "",
    "answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "ROUTE:",
    result["route"]
)

print(
    "ANSWER:",
    result["answer"]
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


KeyError: 'rag_answer'

In [ ]:
initial_state = {
    "question": (
        "What is Maybank's RSI "
        "and what does it indicate?"
    ),
    "route": "",
    "answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "ROUTE:",
    result["route"]
)

print(
    "\nANSWER:"
)

print(
    result["answer"]
)

ROUTE: market

ANSWER:
On 2026-11-06, Maybank's 14-day Relative Strength Index (RSI) was 55.61.

The RSI is a momentum oscillator that measures the speed and change of price movements. A value of 55.61 indicates that Maybank is currently in neutral territory, leaning slightly towards being overbought, but not yet in the commonly accepted overbought range (typically above 70). It suggests that the stock is neither overbought nor oversold based on recent price action.


In [ ]:
result = forecast_stock("Predict Maybank's next-day return.")
print(result)

{'company': 'Maybank', 'date': '2026-09-14', 'predicted_return': -0.06406696140766144, 'direction': 'DOWN', 'answer': "Based on the latest 60 trading sessions available through 2026-09-14, the LSTM forecasting model estimates Maybank's next-session return at -0.06%. The estimated direction is DOWN. This is a model-based quantitative signal, not a guaranteed market outcome."}


In [ ]:
initial_state = {
    "question": (
        "Predict Maybank's next-day return."
    ),
    "route": "",
    "answer": ""
}


result = graph.invoke(
    initial_state
)


print(
    "ROUTE:",
    result["route"]
)

print(
    "\nANSWER:"
)

print(
    result["answer"]
)

ROUTE: forecast

ANSWER:
Based on the latest 60 trading sessions available through 2026-09-14, the LSTM forecasting model estimates Maybank's next-session return at -0.06%. The estimated direction is DOWN. This is a model-based quantitative signal, not a guaranteed market outcome.
